# Stage 4: the matrix as an `OperatorMatrix`

A `LinearOperator` is basis-independent, so it has no matrix representation of its own. Choosing a
basis fixes the matrix representation, and `OperatorMatrix(basis, operator)` builds it.

Constructing an `OperatorMatrix` computes no entries. The matrix is usually far too large to hold,
so `OperatorMatrix` holds no part of it. `matrix[bra, ket]` computes one entry, and only that entry.

That is why the size of the grid never enters the work.

The cells below use KPP–Fisher on four sites,
$\dot u_i=D(u_{i-1}-2u_i+u_{i+1})/h^{2}+Qu_i+Ru_i^{2}$.

In [1]:
import sympy as sp

from lin_ode.basis import (
    CarlemanBasis,
    GaussianHermiteBasis,
    IncompatibleBasisError,
    PSCarlemanBasis,
)
from lin_ode.operators import Koopman, Liouville
from lin_ode import OperatorMatrix, OrdinaryDifferentialSystem

D, Q, R, h = sp.symbols("D Q R h")
u, i = sp.IndexedBase("u"), sp.Idx("i", (0, 3))
fisher_kpp = D * (u[i - 1] - 2 * u[i] + u[i + 1]) / h**2 + Q * u[i] + R * u[i] ** 2

ode = OrdinaryDifferentialSystem.from_sympy(fisher_kpp, u, (i,))
matrix = OperatorMatrix(CarlemanBasis(4, degree=2), Koopman().create_operator(ode))

A `(site, power)` label names a basis state: `(2, 1)` is $u_2$, `((1,1), (2,1))` is $u_1u_2$.

How the value is computed belongs to the basis. The operator applies the basis's multiplication and
differentiation rules and reads the target coefficient.

The right-hand side of the KPP–Fisher equation has three kinds of term: a neighbour (hop), the same
site (on-site), and the nonlinear reaction. The next cell asks for one bra of each against the same
ket, and for a site the equation never reaches.

In [2]:
on_site, hop, reaction, far = (2, 1), (1, 1), (2, 2), (0, 1)

for name, bra in [("on-site", on_site), ("hop", hop), ("reaction", reaction), ("far", far)]:
    print(f"{name:9} < {bra} | L | u_2 > =", matrix[bra, on_site])

on-site   < (2, 1) | L | u_2 > = -2*D/h**2 + Q
hop       < (1, 1) | L | u_2 > = D/h**2
reaction  < (2, 2) | L | u_2 > = R
far       < (0, 1) | L | u_2 > = 0


## Allowed pairs of method and basis

| linearization \ basis | `CarlemanBasis` | `PSCarlemanBasis` | `GaussianHermiteBasis` |
|---|---|---|---|
| `Koopman` | ✓ | ✓ | ✓ |
| `Liouville` | ✗ | ✗ | ✓ |
| `KoopmanVonNeumann` | ✗ | ✗ | ✓ |

✗ = the method transports a density $\rho$, so its basis functions must lie in $L^2(\mathbb{R}^N)$.
Carleman monomials do not.

`OperatorMatrix` refuses a ✗ pair when you construct it, not when you ask for an entry.

In [3]:
liouville = Liouville().create_operator(ode)

OperatorMatrix(GaussianHermiteBasis(4, degree=2), liouville)
print("GaussianHermiteBasis + Liouville: built")

try:
    OperatorMatrix(CarlemanBasis(4, degree=2), liouville)
except IncompatibleBasisError as error:
    print(error)

GaussianHermiteBasis + Liouville: built
Liouville requires an L2(R^N) basis, but CarlemanBasis is not in L2, so the expansion is undefined


## The same label on every basis

Changing basis works the same way. Every basis uses the same `(site, power)` labels, so the slice
you ask for does not change. Only the rules that evaluate the slice change, and therefore the
value.

In [4]:
pivot = sp.Symbol("s")

for basis in (CarlemanBasis(4, 2), PSCarlemanBasis(4, 2, pivot), GaussianHermiteBasis(4, 2)):
    matrix = OperatorMatrix(basis, Koopman().create_operator(ode))
    print(f"{type(basis).__name__:22} < u_2^2 | L | u_2 > =", matrix[reaction, on_site])

CarlemanBasis          < u_2^2 | L | u_2 > = R
PSCarlemanBasis        < u_2^2 | L | u_2 > = R
GaussianHermiteBasis   < u_2^2 | L | u_2 > = -2*R


`PSCarlemanBasis` agrees with `CarlemanBasis` because $u_2$ is inside the basis. The two differ only
where the fold applies.

## Cost at any grid size and any degree

Neither the number of sites nor the truncation degree enters the work. What does is how many sites
a state occupies.

In [5]:
N = 1_000_000_000
k = sp.Idx("k", (0, N - 1))
big = OrdinaryDifferentialSystem.from_sympy(
    D * (u[k - 1] - 2 * u[k] + u[k + 1]) / h**2 + Q * u[k] + R * u[k] ** 2, u, (k,)
)

large = OperatorMatrix(GaussianHermiteBasis(N, degree=6), Liouville().create_operator(big))

print("reaction   < vac | L | u_5 > =", large[(), (5, 1)])
print("divergence < u_5 | L | u_5 > =", large[(5, 1), (5, 1)])

reaction   < vac | L | u_5 > = -3*sqrt(2)*R/4
divergence < u_5 | L | u_5 > = 1000000000*D/h**2 - 500000000*Q


$N$ appears in a value wherever the equation sums over every site. A Liouville generator, for
example, contains $-\nabla\!\cdot\!f$, which is such a sum, so $N$ appears in the diagonal entries
and in no others. Computing either entry costs the same whatever $N$ is.

In [6]:
operator = Koopman().create_operator(big)
L10 = OperatorMatrix(CarlemanBasis(N, degree=10), operator)

u5_10 = (5, 10)                         # u_5**10 -- one site, total degree 10
L10[u5_10, u5_10]                       # < u_5**10 | L | u_5**10 >

-20*D/h**2 + 10*Q

A diagonal entry is $|\boldsymbol\alpha|\,(Q-2D/h^{2})$, so this one is ten times the degree-1
value. Computing it costs what a degree-3 state costs, because $u_5^{10}$ occupies one site.

## Blocks from lists of labels

`matrix[bras, kets]` takes a list of labels on each side and returns the submatrix on those labels.

The cell below takes a $6\times6$ block from the same billion-site system. That costs 36 element
solves, and the size of the matrix does not enter.

In [7]:
L = OperatorMatrix(CarlemanBasis(N, degree=3), operator)

# states on sites {5, 6} of total degree <= 2, by their (site, power) labels
labels = [(), (5, 1), (6, 1), (5, 2), ((5, 1), (6, 1)), (6, 2)]
L[labels, labels].subs({h: 1, D: sp.Rational(1, 20), Q: 0, R: 1})

Matrix([
[0,     0,     0,    0,    0,    0],
[0, -1/10,  1/20,    0,    0,    0],
[0,  1/20, -1/10,    0,    0,    0],
[0,     1,     0, -1/5, 1/20,    0],
[0,     0,     0, 1/10, -1/5, 1/10],
[0,     0,     1,    0, 1/20, -1/5]])

The diagonal follows the same formula as above, and the off-diagonals are the coefficients of
$f_5$. Both follow from the Carleman rules.

## An observable as an `OperatorMatrix`

Under `KoopmanVonNeumann` an observable becomes an operator, and a basis turns that operator into an
`OperatorMatrix`, exactly as for a generator. `Koopman` and `Liouville` hold an observable as a
vector of coefficients instead ([`06_vector`](06_vector.ipynb)).

`ObservableExpression` holds the function of the state, and `get_matrix(basis)` returns the
`OperatorMatrix` directly. You index the result with two labels, exactly as above, and the entry is
$\langle\beta\rvert g\lvert\alpha\rangle$.

In [8]:
from lin_ode import ObservableExpression

energy = ObservableExpression.from_sympy(sp.Sum(u[i] ** 2, (i, 0, 3)).doit(), u, (i,))
G = energy.get_matrix(GaussianHermiteBasis(4, degree=2))

print("observable               ", energy.expr)
print("< vac | g | vac >        ", G[(), ()])
print("< psi_2(u_2) | g | vac > ", G[((2, 2),), ()])

observable                u[0]**2 + u[1]**2 + u[2]**2 + u[3]**2
< vac | g | vac >         2


< psi_2(u_2) | g | vac >  sqrt(2)/2


The example [`kvn_heisenberg`](../examples/kvn_heisenberg.ipynb) carries kinetic and potential
energy through a Koopman–von Neumann evolution, with each observable as a matrix.

## Summary

```python
matrix = OperatorMatrix(basis, method.create_operator(ode))
matrix[bra, ket]          # one element
matrix[bras, kets]        # a block, from two lists of labels
```

- A basis state is its `(site, power)` factors. Every basis uses the same labels, so changing
  basis changes only the value.
- `OperatorMatrix` is lazy, so the cost of an element does not depend on $N$.
- A block is $\dim^2$ element solves, so whole-basis blocks suit small bases.

The example [`kpp_fisher_1d`](../examples/kpp_fisher_1d.ipynb) fills a $9139\times9139$ matrix from
88 element calls, one per distinct label.

---

[← GaussianHermiteBasis](04_basis/gaussian_hermite.ipynb) · [Notebook index](../README.md) · [5. Vectors →](06_vector.ipynb)